In [ ]:
"""
Servidor MCP para Anamnesis Médica Básica
Incluye salvaguardas éticas y técnicas apropiadas
"""

import asyncio
import json
from typing import Dict, List, Optional, Any
from dataclasses import dataclass
from enum import Enum

from mcp.server.fastmcp import FastMCP, Context
from mcp.server.session import ServerSession
from pydantic import BaseModel, Field


class ConsentStatus(Enum):
    PENDING = "pending"
    ACCEPTED = "accepted"
    DECLINED = "declined"


class PatientData(BaseModel):
    """Estructura de datos del paciente para anamnesis"""
    patient_id: str
    consent_status: ConsentStatus
    chief_complaint: Optional[str] = None
    symptom_duration: Optional[str] = None
    symptom_severity: Optional[int] = Field(None, ge=1, le=10)
    associated_symptoms: List[str] = []
    medical_history: List[str] = []
    family_history: List[str] = []
    medications: List[str] = []
    allergies: List[str] = []
    lifestyle_factors: Dict[str, Any] = {}


class AnamnesisProbability(BaseModel):
    """Resultado de clasificación médica"""
    condition: str
    probability: float = Field(ge=0.0, le=1.0)
    confidence_level: str
    supporting_symptoms: List[str]


class MedicalClassificationResult(BaseModel):
    """Resultado completo de clasificación"""
    patient_id: str
    top_conditions: List[AnamnesisProbability]
    disclaimer: str
    recommendation: str
    timestamp: str


# Simulación de almacenamiento (en producción usar DynamoDB)
patient_sessions: Dict[str, PatientData] = {}

mcp = FastMCP(
    name="Medical Anamnesis Agent",
    instructions="""
    Agente especializado en recolección de anamnesis médica básica.
    
    IMPORTANTE LEER: Esta herramienta es solo para fines informativos y educativos.
    NO reemplaza el diagnóstico médico profesional. Siempre consulte a un médico.
    """,
)


@mcp.tool()
async def request_consent(
    patient_id: str,
    ctx: Context[ServerSession, None]
) -> Dict[str, str]:
    """
    Solicita consentimiento informado al paciente.
    
    Args:
        patient_id: ID único del paciente
    """
    
    consent_text = """
    CONSENTIMIENTO INFORMADO PARA ANAMNESIS DIGITAL
    
    Esta herramienta recolecta información médica básica con fines informativos únicamente.
    
    IMPORTANTE:
    - Este NO es un diagnóstico médico
    - Los resultados NO reemplazan la consulta médica profesional
    - Siempre consulte a un médico calificado
    - Sus datos serán procesados de forma segura y anónima
    
    ¿Acepta estos términos? (Responda 'acepto' o 'no acepto')
    """
    
    # Inicializar sesión del paciente
    patient_sessions[patient_id] = PatientData(
        patient_id=patient_id,
        consent_status=ConsentStatus.PENDING
    )
    
    await ctx.info(f"Solicitando consentimiento para paciente {patient_id}")
    
    return {
        "consent_text": consent_text,
        "patient_id": patient_id,
        "status": "pending_consent"
    }


@mcp.tool()
async def process_consent_response(
    patient_id: str,
    response: str,
    ctx: Context[ServerSession, None]
) -> Dict[str, str]:
    """
    Procesa la respuesta de consentimiento del paciente.
    """
    
    if patient_id not in patient_sessions:
        return {"error": "por eticaa no se puede acceder"}
    
    patient = patient_sessions[patient_id]
    
    if "acepto" in response.lower():
        patient.consent_status = ConsentStatus.ACCEPTED
        await ctx.info(f"Consentimiento aceptado para paciente {patient_id}")
        
        return {
            "status": "consent_accepted",
            "message": "Consentimiento aceptado. Procediendo con la anamnesis.",
            "next_step": "chief_complaint"
        }
    else:
        patient.consent_status = ConsentStatus.DECLINED
        await ctx.warning(f"Consentimiento rechazado para paciente {patient_id}")
        
        return {
            "status": "consent_declined",
            "message": "Consentimiento no otorgado. No se puede proceder con la anamnesis.",
            "recommendation": "Si tiene preocupaciones médicas, consulte directamente a un profesional de la salud."
        }


@mcp.tool()
async def collect_chief_complaint(
    patient_id: str,
    complaint: str,
    ctx: Context[ServerSession, None]
) -> Dict[str, str]:
    """
    Recolecta el motivo principal de consulta.
    """
    
    if patient_id not in patient_sessions:
        return {"error": "Sesión de paciente no encontrada"}
    
    patient = patient_sessions[patient_id]
    
    if patient.consent_status != ConsentStatus.ACCEPTED:
        return {"error": "Consentimiento requerido antes de proceder"}
    
    patient.chief_complaint = complaint
    await ctx.info(f"Motivo de consulta registrado para paciente {patient_id}")
    
    return {
        "status": "chief_complaint_recorded",
        "message": f"Motivo registrado: {complaint}",
        "next_step": "symptom_details"
    }


@mcp.tool()
async def collect_symptom_details(
    patient_id: str,
    duration: str,
    severity: int,
    associated_symptoms: List[str],
    ctx: Context[ServerSession, None]
) -> Dict[str, Any]:
    """
    Recolecta detalles específicos de los síntomas.
    """
    
    if patient_id not in patient_sessions:
        return {"error": "Sesión de paciente no encontrada"}
    
    patient = patient_sessions[patient_id]
    
    if patient.consent_status != ConsentStatus.ACCEPTED:
        return {"error": "Consentimiento requerido antes de proceder"}
    
    # Validar severidad
    if not (1 <= severity <= 10):
        return {"error": "La severidad debe estar entre 1 y 10"}
    
    patient.symptom_duration = duration
    patient.symptom_severity = severity
    patient.associated_symptoms = associated_symptoms
    
    await ctx.info(f"Detalles de síntomas registrados para paciente {patient_id}")
    
    return {
        "status": "symptom_details_recorded",
        "summary": {
            "duration": duration,
            "severity": severity,
            "associated_symptoms": associated_symptoms
        },
        "next_step": "medical_history"
    }


@mcp.tool()
async def collect_medical_history(
    patient_id: str,
    medical_history: List[str],
    family_history: List[str],
    medications: List[str],
    allergies: List[str],
    ctx: Context[ServerSession, None]
) -> Dict[str, Any]:
    """
    Recolecta antecedentes médicos y familiares.
    """
    
    if patient_id not in patient_sessions:
        return {"error": "Sesión de paciente no encontrada"}
    
    patient = patient_sessions[patient_id]
    
    if patient.consent_status != ConsentStatus.ACCEPTED:
        return {"error": "Consentimiento requerido antes de proceder"}
    
    patient.medical_history = medical_history
    patient.family_history = family_history
    patient.medications = medications
    patient.allergies = allergies
    
    await ctx.info(f"Antecedentes médicos registrados para paciente {patient_id}")
    
    return {
        "status": "medical_history_recorded",
        "summary": {
            "medical_history": medical_history,
            "family_history": family_history,
            "medications": medications,
            "allergies": allergies
        },
        "next_step": "lifestyle_factors"
    }


@mcp.tool()
async def collect_lifestyle_factors(
    patient_id: str,
    smoking: bool,
    alcohol_consumption: str,
    exercise_frequency: str,
    sleep_hours: int,
    stress_level: int,
    ctx: Context[ServerSession, None]
) -> Dict[str, Any]:
    """
    Recolecta factores de estilo de vida relevantes.
    """
    
    if patient_id not in patient_sessions:
        return {"error": "Sesión de paciente no encontrada"}
    
    patient = patient_sessions[patient_id]
    
    if patient.consent_status != ConsentStatus.ACCEPTED:
        return {"error": "Consentimiento requerido antes de proceder"}
    
    patient.lifestyle_factors = {
        "smoking": smoking,
        "alcohol_consumption": alcohol_consumption,
        "exercise_frequency": exercise_frequency,
        "sleep_hours": sleep_hours,
        "stress_level": stress_level
    }
    
    await ctx.info(f"Factores de estilo de vida registrados para paciente {patient_id}")
    
    return {
        "status": "lifestyle_factors_recorded",
        "summary": patient.lifestyle_factors,
        "next_step": "generate_structured_data"
    }


@mcp.tool()
async def generate_structured_anamnesis(
    patient_id: str,
    ctx: Context[ServerSession, None]
) -> Dict[str, Any]:
    """
    Genera la anamnesis estructurada en formato JSON.
    """
    
    if patient_id not in patient_sessions:
        return {"error": "Sesión de paciente no encontrada"}
    
    patient = patient_sessions[patient_id]
    
    if patient.consent_status != ConsentStatus.ACCEPTED:
        return {"error": "Consentimiento requerido antes de proceder"}
    
    # Convertir a diccionario JSON
    structured_data = patient.dict()
    
    await ctx.info(f"Anamnesis estructurada generada para paciente {patient_id}")
    
    return {
        "status": "anamnesis_completed",
        "structured_data": structured_data,
        "next_step": "medical_classification"
    }


@mcp.tool()
async def simulate_medical_classification(
    patient_id: str,
    ctx: Context[ServerSession, None]
) -> MedicalClassificationResult:
    """
    Simula clasificación médica usando IA.
    En producción: conectar a SageMaker/Bedrock o modelo de Hugging Face.
    """
    
    if patient_id not in patient_sessions:
        return {"error": "Sesión de paciente no encontrada"}
    
    patient = patient_sessions[patient_id]
    
    if patient.consent_status != ConsentStatus.ACCEPTED:
        return {"error": "Consentimiento requerido antes de proceder"}
    
    await ctx.info(f"Ejecutando clasificación médica para paciente {patient_id}")
    
    # Simulación de resultados (en producción: llamar a modelo real)
    mock_results = MedicalClassificationResult(
        patient_id=patient_id,
        top_conditions=[
            AnamnesisProbability(
                condition="Infección respiratoria alta",
                probability=0.75,
                confidence_level="Alta",
                supporting_symptoms=["tos", "dolor de garganta", "fatiga"]
            ),
            AnamnesisProbability(
                condition="Síndrome gripal",
                probability=0.65,
                confidence_level="Media",
                supporting_symptoms=["fiebre", "dolores corporales"]
            ),
            AnamnesisProbability(
                condition="Alergia estacional",
                probability=0.30,
                confidence_level="Baja",
                supporting_symptoms=["congestión nasal"]
            )
        ],
        disclaimer="""
        IMPORTANTE: Estos resultados son estimaciones algorítmicas basadas en 
        patrones de datos y NO constituyen un diagnóstico médico. La interpretación 
        y el diagnóstico definitivo deben ser realizados exclusivamente por un 
        médico calificado.
        """,
        recommendation="""
        Se recomienda encarecidamente consultar a un médico para:
        1. Evaluación clínica completa
        2. Examen físico apropiado
        3. Pruebas diagnósticas si son necesarias
        4. Plan de tratamiento personalizado
        """,
        timestamp="2024-01-01T10:00:00Z"
    )
    
    await ctx.info(f"Clasificación completada para paciente {patient_id}")
    
    return mock_results


@mcp.resource("anamnesis://patient/{patient_id}")
async def get_patient_anamnesis(patient_id: str) -> str:
    """
    Obtiene la anamnesis completa de un paciente.
    """
    
    if patient_id not in patient_sessions:
        return json.dumps({"error": "Paciente no encontrado"})
    
    patient = patient_sessions[patient_id]
    return json.dumps(patient.dict(), indent=2, ensure_ascii=False)


@mcp.prompt()
async def anamnesis_interview_guide(
    interview_stage: str = "initial"
) -> str:
    """
    Guía de preguntas para diferentes etapas de la anamnesis.
    """
    
    guides = {
        "initial": """
        Bienvenido a la herramienta de anamnesis digital.
        
        Primero necesito su consentimiento informado antes de proceder.
        Esta herramienta NO reemplaza la consulta médica profesional.
        """,
        
        "chief_complaint": """
        Cuénteme brevemente cuál es su principal preocupación o síntoma 
        que lo trae hoy a consulta.
        """,
        
        "symptom_details": """
        Sobre su síntoma principal:
        - ¿Cuándo comenzó? (hace horas, días, semanas)
        - En una escala de 1-10, ¿qué tan intenso es?
        - ¿Hay otros síntomas acompañantes?
        """,
        
        "medical_history": """
        Información sobre sus antecedentes:
        - ¿Tiene alguna condición médica conocida?
        - ¿Alguien en su familia ha tenido problemas similares?
        - ¿Qué medicamentos toma actualmente?
        - ¿Tiene alergias conocidas?
        """,
        
        "lifestyle": """
        Sobre su estilo de vida:
        - ¿Fuma o ha fumado?
        - ¿Consume alcohol? ¿Con qué frecuencia?
        - ¿Hace ejercicio regularmente?
        - ¿Cuántas horas duerme por noche?
        - ¿Cómo calificaría su nivel de estrés?
        """
    }
    
    return guides.get(interview_stage, "Etapa de entrevista no reconocida.")


if __name__ == "__main__":
    # Para desarrollo local
    mcp.run()